In [1]:
from scipy.spatial import distance
from imutils import face_utils
from pygame import mixer
from collections import deque
import imutils
import dlib
import cv2
import numpy as np

mixer.init()
mixer.music.load("music.wav")

def eye_aspect_ratio(eye):
    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])
    return (A + B) / (2.0 * C)

# ─── Config ───────────────────────────────────────────
CALIB_FRAMES     = 100     # frames to collect during calibration (~3-4 sec)
THRESH_RATIO     = 0.75    # threshold = baseline × this ratio
FRAME_CHECK      = 20      # consecutive frames below thresh to trigger alert
EAR_HISTORY_SIZE = 10      # smoothing window

# ─── State ────────────────────────────────────────────
calibration_ears  = []          # EAR samples collected during calibration
baseline_ear      = None        # computed after calibration
thresh            = None        # auto-set threshold
ear_history       = deque(maxlen=EAR_HISTORY_SIZE)
flag              = 0
calibrated        = False

# ─── Detector setup ───────────────────────────────────
detect  = dlib.get_frontal_face_detector()
predict = dlib.shape_predictor("models/shape_predictor_68_face_landmarks.dat")
(lStart, lEnd) = face_utils.FACIAL_LANDMARKS_68_IDXS["left_eye"]
(rStart, rEnd) = face_utils.FACIAL_LANDMARKS_68_IDXS["right_eye"]

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = imutils.resize(frame, width=450)
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    subjects = detect(gray, 0)

    # ── CALIBRATION PHASE ─────────────────────────────
    if not calibrated:
        progress = len(calibration_ears)
        bar_pct  = int((progress / CALIB_FRAMES) * 200)

        # Instructions
        cv2.putText(frame, "CALIBRATION: Keep eyes open normally",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)

        # Progress bar background
        cv2.rectangle(frame, (10, 50), (210, 70), (50, 50, 50), -1)
        # Progress bar fill
        cv2.rectangle(frame, (10, 50), (10 + bar_pct, 70), (0, 200, 255), -1)
        cv2.putText(frame, f"{int(progress/CALIB_FRAMES*100)}%",
                    (215, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        for subject in subjects:
            shape     = predict(gray, subject)
            shape     = face_utils.shape_to_np(shape)
            leftEye   = shape[lStart:lEnd]
            rightEye  = shape[rStart:rEnd]
            ear       = (eye_aspect_ratio(leftEye) + eye_aspect_ratio(rightEye)) / 2.0

            # Eye size measurement (bounding box area)
            lx, ly, lw, lh = cv2.boundingRect(leftEye)
            rx, ry, rw, rh = cv2.boundingRect(rightEye)
            avg_eye_w = (lw + rw) / 2
            avg_eye_h = (lh + rh) / 2

            # Draw eye contours during calibration
            cv2.drawContours(frame, [cv2.convexHull(leftEye)],  -1, (0, 255, 255), 1)
            cv2.drawContours(frame, [cv2.convexHull(rightEye)], -1, (0, 255, 255), 1)

            # Show live measurements
            cv2.putText(frame, f"Live EAR: {ear:.3f}",
                        (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)
            cv2.putText(frame, f"Eye size: {avg_eye_w:.1f} x {avg_eye_h:.1f} px",
                        (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)

            if ear > 0.15:  # Filter out blinks during calibration
                calibration_ears.append(ear)

        # ── Calibration complete ──
        if len(calibration_ears) >= CALIB_FRAMES:
            calibration_ears_arr = np.array(calibration_ears)

            baseline_ear = float(np.mean(calibration_ears_arr))
            ear_std      = float(np.std(calibration_ears_arr))
            thresh       = baseline_ear * THRESH_RATIO

            # Clamp threshold to a safe range
            thresh = max(0.18, min(thresh, 0.30))

            calibrated   = True
            print(f"\n✅ Calibration complete!")
            print(f"   Baseline EAR : {baseline_ear:.4f}")
            print(f"   EAR Std Dev  : {ear_std:.4f}")
            print(f"   Auto Thresh  : {thresh:.4f}")

    # ── DETECTION PHASE ───────────────────────────────
    else:
        for subject in subjects:
            shape     = predict(gray, subject)
            shape     = face_utils.shape_to_np(shape)
            leftEye   = shape[lStart:lEnd]
            rightEye  = shape[rStart:rEnd]
            leftEAR   = eye_aspect_ratio(leftEye)
            rightEAR  = eye_aspect_ratio(rightEye)
            ear       = (leftEAR + rightEAR) / 2.0

            # Smooth EAR over last N frames
            ear_history.append(ear)
            smooth_ear = np.mean(ear_history)

            # Draw eye contours
            cv2.drawContours(frame, [cv2.convexHull(leftEye)],  -1, (0, 255, 0), 1)
            cv2.drawContours(frame, [cv2.convexHull(rightEye)], -1, (0, 255, 0), 1)

            # ── HUD ──
            cv2.putText(frame, f"EAR:      {smooth_ear:.3f}",
                        (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
            cv2.putText(frame, f"Baseline: {baseline_ear:.3f}",
                        (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)
            cv2.putText(frame, f"Thresh:   {thresh:.3f}",
                        (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 200, 255), 1)

            # EAR visual bar
            bar_len = int(smooth_ear * 400)
            thresh_x = int(thresh * 400)
            cv2.rectangle(frame, (10, 75), (10 + bar_len, 88),
                          (0, 255, 0) if smooth_ear >= thresh else (0, 0, 255), -1)
            cv2.line(frame, (10 + thresh_x, 72), (10 + thresh_x, 91), (0, 200, 255), 2)
            cv2.putText(frame, "T", (10 + thresh_x - 4, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 200, 255), 1)

            # ── Drowsiness check ──
            if smooth_ear < thresh:
                flag += 1
                if flag >= FRAME_CHECK:
                    cv2.putText(frame, "*** DROWSY ALERT ***",
                                (60, 340), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                    if not mixer.music.get_busy():
                        mixer.music.play()
            else:
                flag = 0

    cv2.imshow("Driver Drowsiness Detection", frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord("q"):
        break
    elif key == ord("r") and calibrated:
        # Press R to recalibrate anytime
        calibration_ears.clear()
        calibrated   = False
        baseline_ear = None
        thresh       = None
        ear_history.clear()
        flag = 0
        print("🔄 Recalibrating...")

cap.release()
cv2.destroyAllWindows()



pygame 2.6.1 (SDL 2.28.4, Python 3.11.15)
Hello from the pygame community. https://www.pygame.org/contribute.html

✅ Calibration complete!
   Baseline EAR : 0.2966
   EAR Std Dev  : 0.0230
   Auto Thresh  : 0.2224
🔄 Recalibrating...

✅ Calibration complete!
   Baseline EAR : 0.3195
   EAR Std Dev  : 0.0308
   Auto Thresh  : 0.2396
